# Populations Distribution
### Import libraries

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

Global paths

In [ ]:
# Current Working Directory
CWD = os.getcwd()
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']

In [ ]:
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))

In [ ]:
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))

from surqct.src.chemical_systems import systems as syst
from surqct.src.postprocessing.MasterEquation import Plotting

Case-specific paths

In [ ]:
# Output directory
out_dir = CWD + '/Output/Populations/em14/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initializing Runs

Global inputs

In [ ]:
SystemName   = 'O3_UMN'
CaseSuffixes = {
    'QCT':        '/oldRelax/em14/',
    'SurQCT-Ref': '/oldRelax/em14.ref/'
    # 'SurQCT-C1':  '/em14.v1/',
    # 'SurQCT-C2':  '/em14.v2/'
}
FigTTranVec  = {
    'PopLowT': [1500, 2500],
    'PopHighT': [8000, 20000]
}
FigTimeShots = {
    'PopLowT': ['50%', '50%'],
    'PopHighT': ['QSS', 'QSS']
}

#### Initialization

Initializing chemical systems

In [ ]:
Systems = {}
for KinSource in CaseSuffixes.keys():
    Systems[KinSource] = syst.get_system(
        NameLong=SystemName,
        KinSource=KinSource.split('-')[0]
    )

### Postprocessing Master Equation Solution

In [ ]:
for KinSource, System in Systems.items():
    System.MasterEquation.read_data(CaseSuffix=CaseSuffixes[KinSource])

### Plotting

In [ ]:
from surqct.src.utilities import utils
matplotlib = utils.import_matplotlib()
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib notebook

Collect data

In [ ]:
def get_time_instants(System, Perc, TTran):
    DSet = System.MasterEquation.Data['2_1_1_0'][str(TTran)]
    X = {k: v for k, v in DSet.items() for Mol in System.Molecules if f'X_{Mol.Name}' in k}
    for name, Xi in X.items():
        MolePerc = np.abs((Xi - Xi[0])/(Xi[0] - Xi[-1]))
        Idx = np.argmin(np.abs(Xi - Perc))
        Time = System.MasterEquation.Data['2_1_1_0'][str(TTran)]['t'][Idx]
    return Time

In [ ]:
Data = {}
for FigName, TTranVec in FigTTranVec.items():
    Data[FigName] = {}
    for KinSource, System in Systems.items():
        Data[FigName][KinSource] = {}
        for i, TTran in enumerate(TTranVec):
            # Time vector
            t  = System.MasterEquation.Data['2_1_1_0'][str(TTran)]['t']
            # Populations evolution
            Ni = System.MasterEquation.Data['2_1_1_0'][str(TTran)]['Ni']
            # Time instant
            Time = FigTimeShots[FigName][i]
            if Time == 'QSS':
                DataQSS = pd.DataFrame.from_records(
                    System.MasterEquation.Data['2_1_1_0']['QSS']
                )
                TimeShot = DataQSS.loc[DataQSS['T'] == TTran]['Mid']
            elif '%' in Time:
                Perc = float(Time.replace("%", ""))/100
                TimeShot = get_time_instants(System, Perc, TTran)
            else:
                TimeShot = Time
            Idx = np.argmin(np.abs(np.log(t[1:]) - np.log(np.array(TimeShot)))) + 1
            # Collect pop distribution
            Data[FigName][KinSource][TTran] = tf.nest.map_structure(lambda x: x[Idx], Ni)

Populations distribution

In [ ]:
def plot_populations(
    Path,
    System,
    Data,
    TTranVec,
    TimeShots,
    FigName=None
):
    for iMol, Mol in enumerate(System.Molecules):

        fig, ax = plt.subplots(2, 1, sharex=True)
        ax = np.array(ax).reshape(-1)

        x = Mol.DiatData['EInt'] - Mol.DissE
        for iTemp in range(len(TTranVec)):
            Temp = int(TTranVec[iTemp])
            for i, (k, d) in enumerate(Data.items()):
                Ni  = d[Temp][Mol.Name] / Mol.g.reshape(-1)
                
                if k == 'QCT':
                    kwargs = dict(label=k, c=COLORS[0], alpha=1.)
                else:
                    kwargs = dict(label=k, c=COLORS[i], alpha=0.3)
                    
                ax[iTemp].scatter([], [], c=kwargs['c'], label=kwargs['label'])
                ax[iTemp].scatter(x, Ni, c=kwargs['c'], s=10, alpha=kwargs['alpha'])
                
            ax[iTemp].set_yscale('log')
            if iTemp == 0:
                ax[iTemp].legend(loc='lower left')

            ax[iTemp].annotate(
                '$T$ = %d K' % Temp,
                xy=(0.7, 0.8),
                xytext=(5, 5),
                xycoords="axes fraction",
                textcoords="offset points",
                va="bottom",
                ha="left",
                fontsize=26
            )

        fig.add_subplot(111, frame_on=False)
        plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel('$\epsilon_{Int}$ [eV]', labelpad=45)
        plt.ylabel('$N_i/g_i$ [m$^{-3}$]', labelpad=70)

        Path = Path + '/' + Mol.Name
        if not os.path.exists(Path):
            os.makedirs(Path)
        if FigName is None:
            FigName = 'Pop'
        plt.savefig(Path + '/' + FigName + '.pdf')
        

In [ ]:
for FigName in Data.keys():
    plot_populations(
        out_dir,
        System,
        Data[FigName],
        FigTTranVec[FigName],
        TimeShotsData[FigName],
        FigName=FigName
    )

Populations distribution error

In [ ]:
def plot_populations_error(
    Path,
    System,
    Data,
    TTranVec,
    TimeShots,
    FigName=None
):
    for iMol, Mol in enumerate(System.Molecules):

        fig, ax = plt.subplots(2, 1, sharex=True, sharey=True)

        x = Mol.DiatData['jqn']
        y = Mol.DiatData['EInt'] - Mol.DissE
        err = []
        for iTemp in range(len(TTranVec)):
            Temp = int(TTranVec[iTemp])
            NiQCT = np.log(Data['QCT'][Temp][Mol.Name])
            for i, (k, d) in enumerate(Data.items()):
                if k == 'QCT':
                    continue
                Ni = np.log(d[Temp][Mol.Name])
                err.append(np.abs((Ni-NiQCT)/NiQCT)*100)

        for iTemp in range(len(TTranVec)):
                
            mappable = ax[iTemp].scatter(x, y, c=err[iTemp], cmap='Reds', s=10, vmin=0., vmax=np.amax(err))
            ax[iTemp].annotate(
                '$T$ = %d K' % int(TTranVec[iTemp]),
                xy=(0.05, 0.8),
                xytext=(3, 3),
                xycoords="axes fraction",
                textcoords="offset points",
                va="bottom",
                ha="left",
                fontsize=26
            )
            ax[iTemp].set_xticks([0,100,200])
    
        fig.add_subplot(111, frame_on=False)
        plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel('J', labelpad=45)
        fig.text(0., 0.41, '$\epsilon_{Int}$ [eV]', va='center', ha='center', rotation='vertical', fontsize=plt.rcParams['axes.labelsize'])
        
        
        cbar = plt.colorbar(
            mappable,
            ax=ax.ravel().tolist(),
            orientation='horizontal',
            location='top',
            aspect=50
        )
        cbar.set_label('Error [\%]', labelpad=20, size=26)

        Path = Path + '/' + Mol.Name
        if not os.path.exists(Path):
            os.makedirs(Path)
        if FigName is None:
            FigName = 'errorPop'
        plt.savefig(Path + '/' + FigName + '.pdf')


In [ ]:
for FigName in Data.keys():
    plot_populations_error(
        out_dir,
        System,
        Data[FigName],
        FigTTranVec[FigName],
        TimeShotsData[FigName],
        FigName='Error'+FigName
    )